# MemoRAG：用“记忆模块”辅助检索的 RAG

MemoRAG 的核心想法是：

- 先把长文档压缩成一套“可检索的记忆”（类似 Key-Value 缓存的抽象）
- 回答问题时，先用记忆生成更好的检索线索（surrogate queries / spans）
- 再用这些线索去检索原始文本片段，最后生成答案

你会在这个 notebook 里实现一个 **简化版 MemoRAG**（保持学习节奏：先跑通最小流程，再做对比）。


In [1]:
import json
import os
from pathlib import Path
from typing import Any

import requests
from dotenv import load_dotenv
from pydantic import BaseModel, Field

import pypdf

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

load_dotenv()

DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

/tmp/ipykernel_2177153/2831976040.py:16: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
import os
os.environ["HTTP_PROXY"] = "http://127.0.0.1:7890"
os.environ["HTTPS_PROXY"] = "http://127.0.0.1:7890"
os.environ["ALL_PROXY"] = "http://127.0.0.1:7890"

## 准备示例数据（PDF + QA）

为了把流程讲清楚，我们沿用原仓库的数据：

- `Understanding_Climate_Change.pdf`
- `q_a.json`（一组问答用于快速对比）

下面统一走“先下载到本地再读取”的方式。

In [3]:
def download_file(url: str, dst: Path, *, timeout_s: float = 60.0) -> Path:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and dst.stat().st_size > 0:
        return dst

    r = requests.get(url, timeout=timeout_s, allow_redirects=True)
    r.raise_for_status()
    dst.write_bytes(r.content)
    return dst


CLIMATE_PDF_URL = "https://raw.githubusercontent.com/NirDiamant/RAG_Techniques/main/data/Understanding_Climate_Change.pdf"
CLIMATE_PDF_PATH = download_file(CLIMATE_PDF_URL, DATA_DIR / "Understanding_Climate_Change.pdf")

QA_URL = "https://raw.githubusercontent.com/NirDiamant/RAG_Techniques/main/data/q_a.json"
QA_PATH = download_file(QA_URL, DATA_DIR / "q_a.json")

str(CLIMATE_PDF_PATH), str(QA_PATH)

('../data/Understanding_Climate_Change.pdf', '../data/q_a.json')

In [ ]:
def load_pdf_pages(file_path: Path, *, max_pages: int | None = None) -> list[Document]:
    reader = pypdf.PdfReader(str(file_path))
    pages = reader.pages
    if max_pages is not None:
        pages = pages[:max_pages]

    docs: list[Document] = []
    for i, page in enumerate(pages):
        text = page.extract_text() or ""
        docs.append(Document(page_content=text, metadata={"source": str(file_path), "page": i}))
    return docs


MAX_PAGES = 8
page_docs = load_pdf_pages(CLIMATE_PDF_PATH, max_pages=MAX_PAGES)
len(page_docs), page_docs[0].page_content[:250]

(8,
 'Understanding Climate Change \nChapter 1: Introduction to Climate Change \nClimate change refers to significant, long-term changes in the global climate. The term \n"global climate" encompasses the planet\'s overall weather patterns, including temperatur')

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200,
)

chunk_docs = text_splitter.split_documents(page_docs)
len(chunk_docs), chunk_docs[0].page_content[:250]

(22,
 'Understanding Climate Change \nChapter 1: Introduction to Climate Change \nClimate change refers to significant, long-term changes in the global climate. The term \n"global climate" encompasses the planet\'s overall weather patterns, including temperatur')

In [ ]:
embeddings = OpenAIEmbeddings()

chunk_vectorstore = FAISS.from_documents(chunk_docs, embeddings)
chunk_retriever = chunk_vectorstore.as_retriever(search_kwargs={"k": 4})

`chunk_vectorstore.as_retriever(...)` 会返回一个 **Retriever**。

在 LangChain 里，Retriever 是 Runnable，因此你后面会看到两种调用方式：

- `chunk_retriever.invoke(query)`：单条检索
- `chunk_retriever.batch([q1, q2, ...])`：批量检索（MemoRAG 用 surrogate queries 时很方便）


## Step 1：构建“记忆模块”（MemoryStore）

记忆模块做两件事：

- **memorize**：把文档 chunk 转成更抽象的 key-value 记忆（topic → details），并存入一个可检索的向量库
- **create_surrogate_queries**：回答问题时，先从记忆里找线索，再让 LLM 生成更适合检索的 surrogate queries


In [ ]:
class KeyValuePair(BaseModel):
    topic: str = Field(description="可检索的主题/实体/问题线索")
    details: str = Field(description="该主题对应的关键信息/细节")


class MemoryResponse(BaseModel):
    pairs: list[KeyValuePair]


class SurrogateQueries(BaseModel):
    queries: list[str]

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

extract_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是一个结构化信息抽取器。请从给定文本中提取关键 topic，并给出对应 details。",
        ),
        (
            "human",
            """文本如下：

{document}

请返回 JSON：{{"pairs": [{{"topic": "...", "details": "..."}}, ...]}}""",
        ),
    ]
)

extract_chain = extract_prompt | llm.with_structured_output(
    MemoryResponse,
    method="json_schema",
    strict=True,
)

surrogate_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是信息检索专家。你会收到：原问题 + 一些从记忆中检索到的线索。你的任务是生成若干条 surrogate queries，用于去原文检索证据。",
        ),
        (
            "human",
            """原问题：{question}

记忆线索：
{memory_hits}

请返回 JSON：{{"queries": ["...", "...", ...]}}。""",
        ),
    ]
)

surrogate_chain = surrogate_prompt | llm.with_structured_output(
    SurrogateQueries,
    method="json_schema",
    strict=True,
)

In [ ]:
class MemoryStore:
    def __init__(self, embeddings: OpenAIEmbeddings):
        self._embeddings = embeddings
        self._vs: FAISS | None = None

    def memorize(self, docs: list[Document]) -> None:
        texts: list[str] = []
        metadatas: list[dict[str, Any]] = []

        for d in docs:
            obj = extract_chain.invoke({"document": d.page_content})
            for kv in obj.pairs:
                text = f"Topic: {kv.topic}\nDetails: {kv.details}"
                texts.append(text)
                metadatas.append({"topic": kv.topic})

        if not texts:
            return

        if self._vs is None:
            self._vs = FAISS.from_texts(texts, self._embeddings, metadatas=metadatas)
        else:
            self._vs.add_texts(texts, metadatas=metadatas)

    def search_memory(self, query: str, *, k: int = 8) -> list[Document]:
        if self._vs is None:
            return []
        return self._vs.similarity_search(query, k=k)

    def create_surrogate_queries(self, question: str, *, k_memory: int = 8) -> list[str]:
        hits = self.search_memory(question, k=k_memory)
        memory_hits = "\n\n".join([h.page_content for h in hits])
        out = surrogate_chain.invoke({"question": question, "memory_hits": memory_hits})
        return [q.strip() for q in out.queries if q.strip()]

    def save_local(self, folder_path: Path) -> None:
        if self._vs is None:
            return
        self._vs.save_local(str(folder_path))

    def load_local(self, folder_path: Path) -> None:
        self._vs = FAISS.load_local(
            str(folder_path),
            self._embeddings,
            allow_dangerous_deserialization=True,
        )

In [10]:
memory_store = MemoryStore(embeddings)

# 为了演示速度，只用前面一小部分 chunks 来“记忆化”
MEMORY_CHUNKS = 6
memory_store.memorize(chunk_docs[:MEMORY_CHUNKS])

# 可选：把记忆库落盘
MEMORY_STORE_DIR = DATA_DIR / "Understanding_Climate_Change_Memory_Store"
memory_store.save_local(MEMORY_STORE_DIR)

str(MEMORY_STORE_DIR)

'../data/Understanding_Climate_Change_Memory_Store'

## Step 2：用记忆生成 surrogate queries，再去原文检索

MemoRAG 的检索与生成可以理解成 2 段：

1. **记忆检索**：在 MemoryStore 里找到与问题相关的 KV 线索
2. **线索→检索**：用 surrogate queries 去 chunk 向量库检索原文证据，再生成答案


In [11]:
answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是一个严谨的问答助手。只能根据给定 context 回答，不要编造。",
        ),
        (
            "human",
            """Context:
{context}

Question:
{question}

请用简洁但足够的信息回答。""",
        ),
    ]
)

answer_chain = answer_prompt | llm


def dedupe_keep_order(items: list[str]) -> list[str]:
    seen: set[str] = set()
    out: list[str] = []
    for x in items:
        if x in seen:
            continue
        seen.add(x)
        out.append(x)
    return out


def memorag_answer(question: str, *, k_per_query: int = 3) -> dict[str, Any]:
    surrogate_queries = memory_store.create_surrogate_queries(question)

    # 用 retriever.batch 并行做多条检索
    docs_lists = chunk_retriever.batch(surrogate_queries)
    ctxs = [d.page_content for docs in docs_lists for d in docs]
    ctxs = dedupe_keep_order(ctxs)

    context = "\n\n".join(ctxs[: (k_per_query * max(1, len(surrogate_queries)))])
    resp = answer_chain.invoke({"context": context, "question": question})
    return {
        "question": question,
        "surrogate_queries": surrogate_queries,
        "context": context,
        "answer": resp.content,
    }


def simple_rag_answer(question: str) -> dict[str, Any]:
    docs = chunk_retriever.invoke(question)
    context = "\n\n".join([d.page_content for d in docs])
    resp = answer_chain.invoke({"context": context, "question": question})
    return {"question": question, "context": context, "answer": resp.content}

In [12]:
question = "What are the impacts of climate change on biodiversity?"

memo_out = memorag_answer(question)
simple_out = simple_rag_answer(question)

{
    "memo_surrogate_queries": memo_out["surrogate_queries"],
    "memo_answer": memo_out["answer"][:300],
    "simple_answer": simple_out["answer"][:300],
}

{'memo_surrogate_queries': ['How does deforestation affect biodiversity and climate change?',
  'What role do human activities play in climate change and its impact on biodiversity?',
  'What are the effects of climate change on tropical rainforests and their biodiversity?',
  'How do modern observations of climate change relate to biodiversity loss?',
  'What is the relationship between agriculture, climate change, and biodiversity?',
  'How does the degradation of boreal forests influence climate change and biodiversity?',
  'What evidence supports the link between human-induced climate change and biodiversity decline?'],
 'memo_answer': 'Climate change impacts biodiversity in several ways:\n\n1. **Habitat Loss**: Changes in temperature and precipitation patterns can alter or destroy habitats, leading to loss of species that cannot adapt or migrate.\n\n2. **Species Extinction**: Increased temperatures and extreme weather events can threa',
 'simple_answer': 'Climate change impacts bi

## Step 3：用一组 QA 做快速对比（Simple RAG vs MemoRAG）

这里我们用 `q_a.json` 里的前几条问题做一个“最小对比”：

- Simple RAG：直接用原问题检索 chunks
- MemoRAG：先用记忆生成 surrogate queries，再检索 chunks

对比的目标是：让你直观看到“surrogate queries 会把检索带到哪里”。

In [13]:
qa = json.loads(QA_PATH.read_text(encoding="utf-8"))

N = 5
questions = [x["question"] for x in qa[:N]]
questions

['What does climate change refer to?',
 "What encompasses the planet's overall weather patterns?",
 'What activities have significantly contributed to climate change over the past century?',
 'How many cycles of glacial advance and retreat have occurred over the past 650,000 years?',
 'What marked the beginning of the modern climate era and human civilization?']

In [14]:
results: list[dict[str, Any]] = []
for q in questions:
    memo = memorag_answer(q)
    simple = simple_rag_answer(q)
    results.append(
        {
            "question": q,
            "memo_surrogate_queries": memo["surrogate_queries"],
            "memo_answer": memo["answer"],
            "simple_answer": simple["answer"],
        }
    )

results[0]

{'question': 'What does climate change refer to?',
 'memo_surrogate_queries': ['What is the definition of climate change?',
  'How do human activities contribute to climate change?',
  'What are the long-term changes associated with climate change?',
  'What role do greenhouse gases play in climate change?',
  'What evidence supports the claim that climate change is driven by human activities?',
  "How has the Earth's climate changed historically?",
  'What are the modern observations related to climate change?',
  'What significant impacts does climate change have on global temperatures and weather patterns?'],
 'memo_answer': '气候变化指的是全球气候的显著、长期变化，包括温度、降水和风模式等天气模式的变化。这些变化主要是由于人类活动，特别是燃烧化石燃料和森林砍伐，导致温室气体的增加。',
 'simple_answer': 'Climate change refers to significant, long-term changes in the global climate, including alterations in temperature, precipitation, and wind patterns. It is primarily driven by human activities, particularly the burning of fossil fuels and deforestation.'}

## Summary

- **MemoRAG 的关键增益点**：先把长文档压缩成可检索的“记忆”（topic/details），再用记忆生成更好的检索线索
- **你需要准备的数据**：原文 chunks 的向量库 + 记忆向量库（记忆由 LLM 抽取 KV 形成）
- **推理时的流程**：question → memory hits → surrogate queries → retrieve chunks → answer

下一步如果你要把这套流程套在自己的数据上，你只需要替换：

- 文档来源（PDF→你的数据源）
- chunk 方式（chunk_size/overlap）
- memorize 的抽取 schema（topic/details 是否够用）
